# Track B agent worker on a Kaggle GPU

Runs the voice agent worker (Whisper STT → intake/LLM → Piper TTS) on this notebook and connects
**outbound** to your LiveKit Cloud project. Kaggle has no inbound ports, so nothing here listens:
the browser and the API on your PC create the room, LiveKit dispatches the job to this worker.

**Before running**
1. Notebook settings → **Accelerator: GPU T4 x2**, **Internet: On** (needs a phone-verified account).
2. Add-ons → **Secrets**, attach: `TRACKB_GEMINI_API_KEY`, `TRACKB_LIVEKIT_URL`,
   `TRACKB_LIVEKIT_API_KEY`, `TRACKB_LIVEKIT_API_SECRET`.
3. **Stop the worker on your PC** — both register as `trackb-intake` in the same project, and
   LiveKit would send each session to whichever it picks.

**When done:** run the last cell (or stop the session) — an idle notebook still burns GPU quota.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
REPO = "/kaggle/working/Rooman-Agent"
!rm -rf {REPO} && git clone -q --depth 1 -b feature/realtime-voice-loop https://github.com/daneshrooman-del/Rooman-Agent.git {REPO}
!cd {REPO} && git log --oneline -1

In [ ]:
# Runtime deps (sentence-transformers is skipped: only knowledge-base ingestion imports it, lazily).
# nvidia-cublas/cudnn: the CUDA libraries ctranslate2 (faster-whisper) loads on the GPU.
!pip install -q "faster-whisper>=1.0" "piper-tts>=1.8" "livekit-agents>=1.8" livekit-api \
    google-genai redis langgraph qdrant-client sqlmodel structlog tenacity pypdf \
    python-multipart pydantic-settings httpx "nvidia-cublas-cu12" "nvidia-cudnn-cu12==9.*"
!pip install -q --no-deps -e {REPO}

In [ ]:
# Redis holds resumable session state (trackb.session.redis_store).
!apt-get -qq install -y redis-server > /dev/null 2>&1; redis-server --daemonize yes > /dev/null; sleep 1; redis-cli ping

In [ ]:
import os, subprocess, urllib.request
MODELS = "/kaggle/working/models/piper"
os.makedirs(MODELS, exist_ok=True)
base = "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/"
for name in ["en_US-lessac-medium.onnx", "en_US-lessac-medium.onnx.json"]:
    path = f"{MODELS}/{name}"
    if not os.path.exists(path):
        urllib.request.urlretrieve(base + name, path)
print(os.listdir(MODELS))

In [ ]:
# Secrets -> environment for the worker process. Values are never printed.
from kaggle_secrets import UserSecretsClient
import nvidia.cublas.lib, nvidia.cudnn.lib

secrets = UserSecretsClient()
env = dict(os.environ)
for name in ["TRACKB_GEMINI_API_KEY", "TRACKB_LIVEKIT_URL",
             "TRACKB_LIVEKIT_API_KEY", "TRACKB_LIVEKIT_API_SECRET"]:
    env[name] = secrets.get_secret(name)

env.update({
    "TRACKB_LLM_PROVIDER": "gemini",
    "TRACKB_GEMINI_MODEL": "gemini-flash-lite-latest",
    "TRACKB_WHISPER_DEVICE": "cuda",
    "TRACKB_WHISPER_COMPUTE_TYPE": "float16",
    "TRACKB_WHISPER_MODEL_SIZE": "small.en",   # GPU can afford better accuracy than CPU tiny.en
    "TRACKB_WHISPER_LANGUAGE": "en",
    "TRACKB_TTS_VOICE_MODEL_PATH": f"{MODELS}/en_US-lessac-medium.onnx",
    "TRACKB_REDIS_URL": "redis://localhost:6379/0",
    "LD_LIBRARY_PATH": ":".join([os.path.dirname(nvidia.cublas.lib.__file__),
                                 os.path.dirname(nvidia.cudnn.lib.__file__),
                                 env.get("LD_LIBRARY_PATH", "")]),
})
print("LiveKit:", env["TRACKB_LIVEKIT_URL"])

In [ ]:
# GPU sanity check in a separate process (same env the worker gets): load Whisper on CUDA and
# time a transcription. If this fails with a cuDNN/cuBLAS error, set TRACKB_WHISPER_DEVICE=cpu
# and TRACKB_WHISPER_COMPUTE_TYPE=int8 above -- Kaggle's CPU is fast enough to keep going.
check = '''
import time, numpy as np
from faster_whisper import WhisperModel
t = time.time(); m = WhisperModel("small.en", device="cuda", compute_type="float16")
print(f"loaded in {time.time()-t:.1f}s")
audio = (np.random.randn(16000 * 5) * 0.01).astype(np.float32)
t = time.time(); list(m.transcribe(audio, language="en")[0]); print(f"5 s of audio in {time.time()-t:.2f}s")
'''
print(subprocess.run(["python", "-c", check], env=env, capture_output=True, text=True, timeout=600).stdout)

In [ ]:
# Start the agent worker (production mode). It connects out to LiveKit Cloud and waits for jobs.
LOG = "/kaggle/working/worker.log"
worker = subprocess.Popen(["python", "-m", "trackb.session.entrypoint", "start"],
                          env=env, cwd=REPO, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
import time; time.sleep(30)
print("worker pid", worker.pid, "running" if worker.poll() is None else f"EXITED {worker.returncode}")
!grep -E "models_prewarmed|registered worker|ERROR" {LOG} | cut -c1-200

Now open the test page on your PC (`http://localhost:8765/voice-test.html`, API running locally
against the same LiveKit Cloud project) and start an intake session. Re-run the cell below to
watch this worker handle it.

In [ ]:
!grep -vE "DEBUG" {LOG} | grep -E "stt_endpoint|stt_transcript|intake_|tts_synth|session_|ERROR" | tail -25 | cut -c1-220

In [ ]:
# Stop the worker (and stop the Kaggle session afterwards to save GPU hours).
worker.terminate(); worker.wait(timeout=30); print("stopped", worker.returncode)